# Linear Regression: Feature Scaling

## Overview

In this demo we follow the **remember - formulate - predict** process, and we use it to answer one question:
**what does feature scaling actually do to a linear regression, and when does it matter?**

We will:

1. **Remember** - load the Student Performance data and *look at the scale of each feature*.
2. **Formulate (raw)** - fit a linear regression on the original, unscaled features.
3. **Formulate (scaled)** - standardize the features and fit the same model again.
4. **Predict & compare** - see what changed, what did not, and why.
5. **Stress test** - find a model where scaling is the difference between working and failing completely.

Along the way, pay close attention to **the order in which we do things**. In machine learning,
doing the right steps in the wrong order is its own category of mistake - and unlike a typo,
it usually produces no error message at all.

**Data:** Student Performance (Kaggle) &nbsp;|&nbsp; **Target:** `Performance Index`

---
## Part 1: Remember - Load and Inspect the Data

First we download the dataset and take a look at it.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np

# Download data from Kaggle
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nikhil7280/student-performance-multiple-linear-regression")
print("Path to dataset files:", path)

# Import dataframe
df = pd.read_csv(f"{path}/Student_Performance.csv")
df

Path to dataset files: /Users/juliadunn/.cache/kagglehub/datasets/nikhil7280/student-performance-multiple-linear-regression/versions/1


,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0
...,...,...,...,...,...,...
9995,1,49,Yes,4,2,23.0
9996,7,64,Yes,8,5,58.0
9997,6,83,Yes,8,5,74.0
9998,9,97,Yes,7,0,95.0


### Preparing the features

One column, `Extracurricular Activities`, is stored as "Yes"/"No" text. A regression model needs
numbers, so we map it to 1/0. Then we separate our **features** (`X`, the inputs) from our
**target** (`y`, the thing we are predicting).

In [2]:
# Convert extracurricular activities to numeric
df["Extracurricular Activities"] = df["Extracurricular Activities"].map({"Yes":1, "No":0})

# Define the features and target variable based on the dataset
feature_vars = ["Hours Studied", "Previous Scores", "Sleep Hours",
                "Sample Question Papers Practiced", "Extracurricular Activities"]
X = pd.DataFrame(df[feature_vars])
y = pd.Series(df["Performance Index"]) # Target: Performance Index

# Display a preview of the dataset
print("Dataset preview:")
print(X.head())
print("\nTarget variable preview:")
print(y.head())

Dataset preview:
   Hours Studied  Previous Scores  Sleep Hours  \
0              7               99            9   
1              4               82            4   
2              8               51            7   
3              5               52            5   
4              7               75            8   

   Sample Question Papers Practiced  Extracurricular Activities  
0                                 1                           1  
1                                 2                           0  
2                                 2                           1  
3                                 2                           1  
4                                 5                           0  

Target variable preview:
0    91.0
1    65.0
2    45.0
3    36.0
4    66.0
Name: Performance Index, dtype: float64


### Now look at the *scales*

Here is the step that most people skip, and the one that motivates this entire notebook.
Before fitting anything, ask: **what units are these features in, and how much does each one vary?**

In [3]:
# Look at the scale of each feature BEFORE we model anything
scales = X.describe().T[["mean", "std", "min", "max"]]
scales["range"] = scales["max"] - scales["min"]

print("Feature scales, in their original units:")
scales.round(2)

Feature scales, in their original units:


,mean,std,min,max,range
Hours Studied,4.99,2.59,1.0,9.0,8.0
Previous Scores,69.45,17.34,40.0,99.0,59.0
Sleep Hours,6.53,1.70,4.0,9.0,5.0
Sample Question Papers Practiced,4.58,2.87,0.0,9.0,9.0
Extracurricular Activities,0.49,0.50,0.0,1.0,1.0


### What that table is telling us

Read down the `std` column - that is how much each feature typically varies:

| Feature | Range | Std. dev. |
|---|---|---|
| Previous Scores | 40 - 99 | **~17.3** |
| Sample Question Papers Practiced | 0 - 9 | ~2.9 |
| Hours Studied | 1 - 9 | ~2.6 |
| Sleep Hours | 4 - 9 | ~1.7 |
| Extracurricular Activities | 0 - 1 | **~0.5** |

`Previous Scores` varies roughly **35 times more** than `Extracurricular Activities`
(17.3 / 0.5). These features do not live on remotely the same numeric scale - one is a test
score out of 100, another is a yes/no flag.

Hold on to that fact. It is about to make the raw model's coefficients very hard to read,
and it is the entire reason feature scaling exists.

---
## Part 2: Split First - Before Anything Else

Before we fit a model (and before we scale anything), we split the data into a **training set**
and a **test set**.

The test set is a stand-in for data the model has never seen. Its whole job is to give us an
honest answer to "how will this perform in the real world?" That honesty only holds if the test
set stays genuinely untouched.

> ### Rule 1: split first.
> The train/test split is the **first** modeling step - before scaling, before imputing,
> before anything else that looks at your data. From this cell onward, treat `X_test`
> as if it does not exist until the moment we call `.predict()`.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score

# SPLIT FIRST - this is always step one
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]:>5} rows")
print(f"Test set:     {X_test.shape[0]:>5} rows")

Training set:  8000 rows
Test set:      2000 rows


---
## Part 3: Formulate - A Baseline Model on Raw Data

Let's fit a linear regression on the **original, unscaled** features. This is our baseline:
whatever scaling does, we want to measure it against this.

In [5]:
# Train on the RAW (unscaled) features
lin_reg_raw = LinearRegression()
lin_reg_raw.fit(X_train, y_train)

# Predict on the raw test data
y_pred_raw = lin_reg_raw.predict(X_test)

# Evaluate
mse_raw  = mean_squared_error(y_test, y_pred_raw)
rmse_raw = root_mean_squared_error(y_test, y_pred_raw)
r2_raw   = r2_score(y_test, y_pred_raw)

print("Raw (Unscaled) Data Model:")
print(f"Mean Squared Error:      {mse_raw:.4f}")
print(f"Root Mean Squared Error: {rmse_raw:.4f}")
print(f"R2 Score:                {r2_raw:.4f}")
print("\nModel Coefficients (Raw):")
print(pd.Series(lin_reg_raw.coef_, index=feature_vars))
print(f"\nIntercept: {lin_reg_raw.intercept_:.4f}")

Raw (Unscaled) Data Model:
Mean Squared Error:      4.0826
Root Mean Squared Error: 2.0206
R2 Score:                0.9890

Model Coefficients (Raw):
Hours Studied                       2.852484
Previous Scores                     1.016988
Sleep Hours                         0.476941
Sample Question Papers Practiced    0.191831
Extracurricular Activities          0.608617
dtype: float64

Intercept: -33.9219


### The problem with these coefficients

The model fits well. But now try to answer a simple question: **which feature matters most?**

Looking at the raw coefficients, `Hours Studied` ~2.85 has a coefficient almost three times larger than `Previous Scores` ~1.02. So studying matters about three times more than prior
performance, right?

**No.** Each coefficient answers a different question, because each feature is in different units:

- `Hours Studied` ~2.85 means: *one additional hour* of study adds ~2.85 points.
- `Previous Scores` ~1.02 means: *one additional point* on the previous score adds ~1.02 points.

One extra hour of study and one extra point on a 40-99 scale are not comparable quantities.
A coefficient's size depends on the unit its feature happens to be measured in, so comparing
raw coefficients across features compares nothing meaningful. Switch `Hours Studied` to minutes
and its coefficient would drop by a factor of 60 - with the model completely unchanged.

This is the problem scaling solves.

---
## Part 4: Scaling the Features

**Standardization** (`StandardScaler`) does not squeeze data into a fixed range. Instead it
re-expresses every feature in the same unit: **standard deviations away from that feature's mean.**

$$z = \frac{x - \mu}{\sigma}$$

After this transformation every feature has mean 0 and standard deviation 1. A value of
$z = 1.5$ means "1.5 standard deviations above average," and that sentence means the same thing
for test scores, study hours, and sleep alike. Now the features are genuinely comparable.

> ## Order of operations: the rule that matters most
>
> A scaler is **not** a formula you simply apply. It is a thing that **learns**: `StandardScaler`
> must estimate a mean ($\mu$) and a standard deviation ($\sigma$) *from data* before it can
> transform anything. That makes the scaler **part of your model** - and it obeys the same rule
> your model does.
>
> | Data | Method to call | What happens |
> |---|---|---|
> | Training set | `.fit_transform(X_train)` | **Learns** $\mu$ and $\sigma$ from the training data, then applies them |
> | Test set | `.transform(X_test)` | Applies the **training set's** $\mu$ and $\sigma$ - learns nothing new |
>
> **Never call `.fit()` or `.fit_transform()` on your test data. Never scale before you split.**
>
> ### Why it matters: data leakage
>
> If you scale the full dataset and split afterward, the $\mu$ and $\sigma$ were computed
> partly from your test rows. Information from the test set has leaked into your training
> pipeline, and your test score is no longer an honest estimate of performance on unseen data.
> This is called **data leakage**.
>
> Here is the dangerous part: **leakage almost never raises an error.** Your code runs. Your
> R2 looks fine - often slightly *better* than it should. Nothing turns red. You find out only
> when the model meets real data and quietly underperforms.
>
> ### This generalizes beyond scalers
>
> **Anything that learns from data is part of the model, and goes after the split.** Scalers,
> imputers (learning a median to fill in), PCA (learning components), encoders that learn a
> category set, feature selectors. If it has a `.fit()` method, it belongs on the training side
> of the line.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Fit the scaler on TRAINING data only, then apply to both
scaler = StandardScaler()

# LEARNS mu and sigma, then transforms
X_train_scaled = scaler.fit_transform(X_train)

# APPLIES the training mu and sigma only
X_test_scaled = scaler.transform(X_test)

### What did the scaler actually learn?

The scaler stored the mean and standard deviation of each feature. Let's look at them directly -
this makes it concrete that the scaler is carrying learned parameters around, exactly like a model.

In [9]:
# The parameters the scaler learned - these came from the TRAINING SET ONLY
learned = pd.DataFrame({
    "mu (training mean)": scaler.mean_,
    "sigma (training std)": scaler.scale_,
}, index=feature_vars)

print("Parameters the scaler learned from X_train:")
print(learned.round(3))

# Compare to the full dataset's statistics - close, but NOT the same numbers
print("\nFor contrast, the FULL dataset's means (train + test together):")
print(pd.Series(X.mean().values, index=feature_vars).round(3))

Parameters the scaler learned from X_train:
                                  mu (training mean)  sigma (training std)
Hours Studied                                  4.983                 2.595
Previous Scores                               69.531                17.343
Sleep Hours                                    6.542                 1.698
Sample Question Papers Practiced               4.616                 2.861
Extracurricular Activities                     0.495                 0.500

For contrast, the FULL dataset's means (train + test together):
Hours Studied                        4.993
Previous Scores                     69.446
Sleep Hours                          6.531
Sample Question Papers Practiced     4.583
Extracurricular Activities           0.495
dtype: float64


Notice that the scaler's $\mu$ values are close to the full dataset's means but **not identical**.
That difference is exactly the information we are deliberately refusing to use. The test set
contributed nothing to these numbers - which is precisely what makes the upcoming test score
trustworthy.

Let's also confirm the transformation did what it claims:

In [10]:
# Verify: scaled TRAINING data has mean 0 and std 1 (by construction)
print("Scaled training data - mean of each feature:")
print(np.round(X_train_scaled.mean(axis=0), 6))
print("\nScaled training data - std of each feature:")
print(np.round(X_train_scaled.std(axis=0), 6))

# The TEST data is close to 0/1 but not exactly - and that is correct!
print("\nScaled TEST data - mean of each feature (NOT exactly 0, and that's right):")
print(np.round(X_test_scaled.mean(axis=0), 6))

Scaled training data - mean of each feature:
[ 0. -0.  0. -0.  0.]

Scaled training data - std of each feature:
[1. 1. 1. 1. 1.]

Scaled TEST data - mean of each feature (NOT exactly 0, and that's right):
[ 0.018596 -0.024593 -0.032457 -0.05802   0.00175 ]


The scaled **training** data has mean exactly 0 and standard deviation exactly 1 - that is what
`fit_transform` guarantees.

The scaled **test** data is close to, but not exactly, 0 and 1. Students often see this and think
something went wrong. It did not. It is the signature of having done this *correctly*: the test
set was transformed using the training set's numbers, so it lands near zero without being forced
there. **If your test data came out at exactly mean 0 and std 1, that would be evidence you had
accidentally re-fit the scaler on it.**

### Now fit the same model on the scaled features

In [11]:
# Train on the SCALED features
lin_reg_scaled = LinearRegression()
lin_reg_scaled.fit(X_train_scaled, y_train)

# Predict on scaled test data
y_pred_scaled = lin_reg_scaled.predict(X_test_scaled)

# Evaluate
mse_scaled  = mean_squared_error(y_test, y_pred_scaled)
rmse_scaled = root_mean_squared_error(y_test, y_pred_scaled)
r2_scaled   = r2_score(y_test, y_pred_scaled)

print("Scaled Data Model:")
print(f"Mean Squared Error:      {mse_scaled:.4f}")
print(f"Root Mean Squared Error: {rmse_scaled:.4f}")
print(f"R2 Score:                {r2_scaled:.4f}")
print("\nModel Coefficients (Scaled):")
print(pd.Series(lin_reg_scaled.coef_, index=feature_vars))
print(f"\nIntercept: {lin_reg_scaled.intercept_:.4f}")

Scaled Data Model:
Mean Squared Error:      4.0826
Root Mean Squared Error: 2.0206
R2 Score:                0.9890

Model Coefficients (Scaled):
Hours Studied                        7.401341
Previous Scores                     17.637271
Sleep Hours                          0.810031
Sample Question Papers Practiced     0.548842
Extracurricular Activities           0.304291
dtype: float64

Intercept: 55.3115


---
## Part 5: Predict - Comparing the Two Models

Let's put the two models side by side and see exactly what scaling changed.

In [12]:
# Performance comparison
performance = pd.DataFrame({
    "Raw (unscaled)": [mse_raw, rmse_raw, r2_raw],
    "Scaled":         [mse_scaled, rmse_scaled, r2_scaled],
}, index=["MSE", "RMSE", "R2"])

print("MODEL PERFORMANCE")
print(performance.round(6))
print(f"\nDifference in R2: {abs(r2_raw - r2_scaled):.10f}")

MODEL PERFORMANCE
      Raw (unscaled)    Scaled
MSE         4.082628  4.082628
RMSE        2.020552  2.020552
R2          0.988983  0.988983

Difference in R2: 0.0000000000


In [13]:
# Coefficient comparison
coef_compare = pd.DataFrame({
    "Raw coef":     lin_reg_raw.coef_,
    "Feature sigma": scaler.scale_,
    "Scaled coef":  lin_reg_scaled.coef_,
}, index=feature_vars)

# Rank features by importance under each version
coef_compare["Rank (raw)"]    = coef_compare["Raw coef"].abs().rank(ascending=False).astype(int)
coef_compare["Rank (scaled)"] = coef_compare["Scaled coef"].abs().rank(ascending=False).astype(int)

print("COEFFICIENT COMPARISON")
print(coef_compare.round(4))

# The exact relationship between them
print("\nCheck -- raw coef x feature sigma should equal the scaled coef:")
print((coef_compare["Raw coef"] * coef_compare["Feature sigma"]).round(4))

COEFFICIENT COMPARISON
                                  Raw coef  Feature sigma  Scaled coef  \
Hours Studied                       2.8525         2.5947       7.4013   
Previous Scores                     1.0170        17.3427      17.6373   
Sleep Hours                         0.4769         1.6984       0.8100   
Sample Question Papers Practiced    0.1918         2.8611       0.5488   
Extracurricular Activities          0.6086         0.5000       0.3043   

                                  Rank (raw)  Rank (scaled)  
Hours Studied                              1              2  
Previous Scores                            2              1  
Sleep Hours                                4              3  
Sample Question Papers Practiced           5              4  
Extracurricular Activities                 3              5  

Check -- raw coef x feature sigma should equal the scaled coef:
Hours Studied                        7.4013
Previous Scores                     17.6373
Sleep H

### What changed, what didn't, and why

**Performance did not change at all.** MSE, RMSE, and R2 are identical out to many decimal
places. This is not a coincidence or a rounding artifact - it is a mathematical guarantee.
Ordinary least squares solves for coefficients exactly, and if you divide a feature by
$\sigma$, the fitting procedure simply multiplies its coefficient by $\sigma$ to compensate.
Every prediction comes out the same. **For plain OLS, scaling cannot improve accuracy.**

That relationship is exactly what the check above confirms:

$$\beta_{\text{scaled}} = \beta_{\text{raw}} \times \sigma$$

**The coefficients changed completely - and so did what they mean.** Look at the two rank
columns. On raw data, `Hours Studied` looks like the most important feature. After scaling,
`Previous Scores` is roughly **2.4 times larger** than `Hours Studied` and is clearly dominant.

**The apparent ranking of the two most important features reverses.** Same model, same
predictions, same R2 - but the raw coefficients told us a genuinely misleading story about
which feature drives performance. The scaled coefficients tell the true one, because they are
all in the same unit: *points of Performance Index per one standard deviation of the feature.*

So for linear regression, scaling is not about accuracy. **It is about being able to read
your own model.**

---
## Part 6: A Fuller Regression Summary

The function below reproduces a `statsmodels`-style OLS summary table so we can inspect standard
errors, t-statistics, p-values, and confidence intervals.

Note how it handles our rule: the **split happens first**, and only then is the scaler fit on
the training portion. The `scaled=True/False` switch lets us run the same analysis both ways.

> **Reading note:** this summary reports the coefficient table from the **training** set (that
> is where the model was fit, so that is where inference on the coefficients belongs), while
> R2, F, AIC, and BIC are computed on the **test** set. Check the label on each row before you
> compare these numbers to output from another package.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

def ols_summary(X, y, lin_model=None, scaled=False, test_size=0.2, random_state=42):

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # Scale if requested
    if scaled:
        scaler = StandardScaler()
        X_train = pd.DataFrame(scaler.fit_transform(X_train),
                               columns=X.columns, index=X_train.index)
        X_test  = pd.DataFrame(scaler.transform(X_test),
                               columns=X.columns, index=X_test.index)

    n_train, p = X_train.shape
    n_test = X_test.shape[0]

    # Fit on train only
    model = lin_model.fit(X_train, y_train)

    # --- Training stats (for inference) ---
    y_pred_train = model.predict(X_train)
    res_train = y_train - y_pred_train
    ss_res_train = np.sum(res_train**2)
    ss_tot_train = np.sum((y_train - np.mean(y_train))**2)

    # --- Test stats (honest performance) ---
    y_pred_test = model.predict(X_test)
    res_test = y_test - y_pred_test
    ss_res_test = np.sum(res_test**2)
    ss_tot_test = np.sum((y_test - np.mean(y_test))**2)

    r2_train = 1 - ss_res_train / ss_tot_train
    r2_test  = 1 - ss_res_test  / ss_tot_test
    r2_adj   = 1 - (1 - r2_test) * (n_test - 1) / (n_test - p - 1)

    # F-statistic (based on test)
    ss_reg = ss_tot_test - ss_res_test
    f_stat = (ss_reg / p) / (ss_res_test / (n_test - p - 1))
    f_pval = 1 - stats.f.cdf(f_stat, p, n_test - p - 1)

    # Log-likelihood / AIC / BIC (based on test)
    log_lik = -n_test/2 * np.log(2 * np.pi * ss_res_test / n_test) - n_test/2
    aic = 2 * (p + 1) - 2 * log_lik
    bic = np.log(n_test) * (p + 1) - 2 * log_lik

    # Coefficient inference (based on train — this is where model was fit)
    MSE = ss_res_train / (n_train - p - 1)
    X_b = np.column_stack([np.ones(n_train), X_train])
    se = np.sqrt(np.diag(MSE * np.linalg.inv(X_b.T @ X_b)))
    coefs = np.concatenate([[model.intercept_], model.coef_])
    t_stats_vals = coefs / se
    p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats_vals), df=n_train - p - 1))
    ci_low  = coefs - 1.96 * se
    ci_high = coefs + 1.96 * se

    names = ["const"] + X.columns.tolist()

    # Print
    print("=" * 78)
    print(f"{'OLS Regression Results':^78}")
    print("=" * 78)
    print(f"{'R-squared (test):':<45}{r2_test:.3f}")
    print(f"{'R-squared (train):':<45}{r2_train:.3f}")
    print(f"{'Adj. R-squared (test):':<45}{r2_adj:.3f}")
    print(f"{'F-statistic:':<45}{f_stat:.3f}")
    print(f"{'Prob (F-statistic):':<45}{f_pval:.5f}")
    print(f"{'Log-Likelihood:':<45}{log_lik:.3f}")
    print(f"{'No. Observations (train/test):':<45}{n_train}/{n_test}")
    print(f"{'AIC:':<45}{aic:.2f}")
    print(f"{'BIC:':<45}{bic:.2f}")
    print(f"{'Df Residuals:':<45}{n_test - p - 1}")
    print(f"{'Df Model:':<45}{p}")
    print("=" * 78)
    print(f"{'':16}{'coef':>10}{'std err':>10}{'t':>10}{'P>|t|':>10}{'[0.025':>10}{'0.975]':>10}")
    print("-" * 78)
    for name, c, s, t, pv, lo, hi in zip(names, coefs, se, t_stats_vals, p_values, ci_low, ci_high):
        print(f"{name:<16}{c:>10.4f}{s:>10.3f}{t:>10.3f}{pv:>10.3f}{lo:>10.3f}{hi:>10.3f}")
    print("=" * 78)

### Summary on scaled features

In [ ]:
ols_summary(X, y, lin_model = LinearRegression(), scaled = True)

                            OLS Regression Results                            
R-squared (test):                            0.989
R-squared (train):                           0.989
Adj. R-squared (test):                       0.989
F-statistic:                                 35800.758
Prob (F-statistic):                          0.00000
Log-Likelihood:                              -4244.618
No. Observations (train/test):               8000/2000
AIC:                                         8501.24
BIC:                                         8534.84
Df Residuals:                                1994
Df Model:                                    5
                      coef   std err         t     P>|t|    [0.025    0.975]
------------------------------------------------------------------------------
const              55.3115     0.023  2421.828     0.000    55.267    55.356
Hours Studied       7.4013     0.023   324.007     0.000     7.357     7.446
Previous Scores    17.6373     0.023 

### Summary on raw features

In [ ]:
ols_summary(X, y, lin_model = LinearRegression(), scaled = False)

                            OLS Regression Results                            
R-squared (test):                            0.989
R-squared (train):                           0.989
Adj. R-squared (test):                       0.989
F-statistic:                                 35800.758
Prob (F-statistic):                          0.00000
Log-Likelihood:                              -4244.618
No. Observations (train/test):               8000/2000
AIC:                                         8501.24
BIC:                                         8534.84
Df Residuals:                                1994
Df Model:                                    5
                      coef   std err         t     P>|t|    [0.025    0.975]
------------------------------------------------------------------------------
const             -33.9219     0.143  -237.380     0.000   -34.202   -33.642
Hours Studied       2.8525     0.009   324.007     0.000     2.835     2.870
Previous Scores     1.0170     0.001 

### Comparing the two summaries

Run both and compare them line by line. Notice what is **identical** between them:

- R-squared (train and test), adjusted R-squared
- F-statistic and its p-value
- Log-likelihood, AIC, BIC
- Every **t-statistic** and every **p-value** in the coefficient table

And notice what **differs**:

- The `coef` column
- The `std err` column
- The confidence interval bounds
- The intercept (`const`)

This is the same lesson as Part 5, now visible in the inference statistics. Scaling rescales
each coefficient and its standard error by the *same* factor $\sigma$, so the ratio between
them - the t-statistic - is untouched. **Statistical significance does not depend on the units
you chose.** A feature that matters still matters whether you measure study time in hours or in
minutes; only the number attached to it changes.

What scaling buys you is that the `coef` column finally becomes readable *across rows*.

---
# Conclusion

In this demo we:

- Inspected the **scale** of each feature before modeling, and found a ~35x spread in variation.
- **Split first**, then fit a baseline linear regression on raw, unscaled features.
- Standardized the features - fitting the scaler on training data **only** - and refit the model.
- Compared the two and found identical predictions but completely different coefficients.
- Showed a model (gradient descent) where scaling is the difference between working and failing.

### Key takeaways

**1. Order of operations is not a style preference.**
Split, then scale. Fit the scaler on training data, transform the test data with it. Anything
with a `.fit()` method is part of your model and belongs on the training side of that line.

**2. Data leakage is silent.**
Scaling before splitting throws no error and produces a plausible - even flattering - R2.
The absence of a red error message is not evidence that a pipeline is correct.

**3. For OLS, scaling changes interpretation, not accuracy.**
Predictions, R2, RMSE, t-statistics, and p-values are all unchanged. What changes is whether
the coefficients can be compared to each other at all.

**4. Raw coefficients can actively mislead.**
On this dataset the raw coefficients made `Hours Studied` look like the dominant predictor.
Scaled, `Previous Scores` is roughly 2.4x larger. A coefficient's magnitude is partly an
accident of the unit it is measured in.

**5. For most other algorithms, scaling is mandatory.**
Gradient descent, k-NN, k-means, SVMs, regularized regression, PCA, and neural networks all
depend on it. Since scaling never hurts, make it your default.

This process reflects the **remember - formulate - predict** approach to machine learning.

---
## Check Your Understanding

1. You run `StandardScaler().fit_transform(X)` on your whole dataset, then call
   `train_test_split`. Your test R2 comes out at 0.94 and no errors appear. What is wrong,
   and why did nothing warn you?

2. You scale your training data and get a test R2 of 0.87. A classmate says they can improve
   it by also calling `fit_transform` on the test set so it is "properly normalized too."
   What actually happens to the reported score, and why is that number no longer meaningful?

3. After scaling, `Previous Scores` has a coefficient of about 17.6. State in one plain English
   sentence what that number means.

4. Your teammate fits a decision tree and a k-nearest-neighbors model on the same unscaled data.
   The tree performs fine; k-NN performs poorly. Explain why scaling matters for one and not the other.

5. You measure `Hours Studied` in minutes instead of hours and refit the raw model. Which of
   these change: the coefficient on `Hours Studied`, its t-statistic, its p-value, the model's R2?